# M3.S2 — Memory Hierarchy, Data Locality & Performance-Aware Data Structures
## Why moving data can cost more than computing on it

This notebook accompanies **M3.S2 — Memory Hierarchy, Data Locality & Data Structures**.

The objective is to move from:

> **“I know where caches and DRAM are.”**

To:

> **“I can predict how a program's access pattern and data layout will interact with the memory system.”**

### Classroom method

> **PREDICT → RUN → OBSERVE → EXPLAIN**

### Two kinds of evidence

- **REAL CLUSTER EXPERIMENT** — measured code running on the SciTech compute resources.
- **MODEL** — a simplified calculation used to reason about performance.

Do not confuse model predictions with measurements.

### Core live route

1. Inspect the memory topology.
2. Compare **latency** and **bandwidth**.
3. Make **cache-line effects** visible with stride.
4. Compare **row-major and column-major traversal**.
5. Expose **false sharing**.
6. Compare **AoS and SoA** for a particle kernel.
7. Use a **qualitative Roofline** to classify memory-bound vs compute-bound work.

### Extensions in the same notebook

- blocking / tiling;
- loop fusion and reduced memory traffic;
- NUMA first-touch placement;
- compiler vectorization evidence;
- sparse CSR storage and irregular access;
- software prefetching;
- locality across cache → NUMA → node.

### Build

**M3S2-2026-09-26-v1**

The notebook uses only the Python standard library for orchestration. The measured kernels are small C/OpenMP programs compiled inside the Slurm allocation.

**Build M3S2-2026-10-04-v2.** Save a personal copy. Choose **Restart Kernel and Run All Cells**. A saved job ID avoids duplicate submissions across restarts. Change one setting in Section 1 and run from there downward. GPU/MPI network timings are not measured here.

The real workloads represent climate preprocessing (fusion), image transpose (tiling), particle motion (AoS/SoA), and a sparse interaction matrix (CSR). These are simplified kernels, not full applications.


# 0 — Read the machine before reasoning about the code

### Predict

Before running the next cell:

- How many cache levels do you expect?
- Is the Jupyter host necessarily the compute node used for timing?
- Does the node expose one NUMA domain or more than one?
- What is the cache-line size?

The goal is **not** to memorize commands. The goal is to connect software behavior to the real machine.


In [ ]:
import os
import platform
import re
import shutil
import subprocess
import time
from pathlib import Path


def run_command(command, max_lines=40):
    print('$', command)
    p = subprocess.run(command, shell=True, executable='/bin/bash', capture_output=True, text=True)
    text = (p.stdout or '') + (('\n[stderr]\n' + p.stderr) if p.stderr.strip() else '')
    lines = text.strip().splitlines()
    if len(lines) > max_lines:
        lines = lines[:max_lines] + [f'... ({len(text.strip().splitlines())-max_lines} more lines)']
    if lines:
        print('\n'.join(lines))
    print('return code:', p.returncode)
    print()
    return p

print('Host:', platform.node())
print('Visible CPUs to this kernel:', os.cpu_count())
print('sbatch:', shutil.which('sbatch'))
print('gcc:', shutil.which('gcc'))
print('numactl:', shutil.which('numactl'))
print()
run_command("lscpu | grep -E 'Architecture|Model name|Socket|Core|Thread|NUMA|L1d|L2|L3'", max_lines=30)
run_command("getconf LEVEL1_DCACHE_LINESIZE 2>/dev/null || true")
if shutil.which('numactl'):
    run_command('numactl --hardware', max_lines=30)
else:
    print('numactl is not installed on this host. The Slurm job will still report lscpu topology.')


### Explain

A core normally does **not** fetch only the single scalar named in your source code. Memory is transferred through the hierarchy in blocks, commonly cache lines.

This session is therefore about the path:

```text
memory hierarchy
      ↓
access pattern
      ↓
data layout
      ↓
data movement
      ↓
performance
```

For timing, we will use **allocated compute resources** rather than treating the Jupyter host as a benchmark machine.


# 1 — Configure and prepare the real memory lab

Edit ONE setting, predict what changes, and run from this section downward. The next cell creates the exact tracked C files in a dedicated folder. It does not compile or run heavy workloads on the Jupyter host.

**All real timings come from one Slurm allocation: 16 CPUs, 4 GB, eight-minute execution limit.** Queue waiting is separate. Smaller settings may fit in cache; larger settings may expose different limits.


In [ ]:
# CHANGE ONE value at a time. Allocated CPUs stay at 16; this controls active workers.
LAB_SETTINGS = {
    "M3_THREADS": 16,              # TRY 1, 4, 8, 16
    "M3_ARRAY_N": 4194304,         # doubles/particles/index entries; TRY 1048576
    "M3_MATRIX_N": 2048,          # square image size; TRY 1024, 2049 (non-divisible tiles)
    "M3_TILE": 32,                # TRY 8, 16, 64, 128
    "M3_COUNTER_ITERS": 2000000,
    "M3_PREFETCH_DISTANCE": 64,   # elements ahead; TRY 8, 256
    "M3_SPARSE_N": 200000,        # rows in the sparse problem
    "M3_NNZ_PER_ROW": 5,          # TRY 3, 9
}
limits={"M3_THREADS":(1,16),"M3_ARRAY_N":(1024,16777216),"M3_MATRIX_N":(16,4096),
        "M3_TILE":(1,512),"M3_COUNTER_ITERS":(1000,20000000),"M3_PREFETCH_DISTANCE":(1,4096),
        "M3_SPARSE_N":(16,1000000),"M3_NNZ_PER_ROW":(1,32)}
for key,value in LAB_SETTINGS.items():
    lo,hi=limits[key]
    if type(value) is not int or not lo<=value<=hi: raise ValueError(f"{key}: integer {lo}..{hi} required")
print("Experiment settings:",LAB_SETTINGS)
DEMO_DIR=Path("m3s2_demo_files_v2").resolve()
DEMO_DIR.mkdir(exist_ok=True)

# These sources match session_demos/12_memory_locality in GitHub.
DEMO_SOURCES={'m3s2_memory_lab.c': '\n#define _GNU_SOURCE\n#include <stdio.h>\n#include <stdlib.h>\n#include <stdint.h>\n#include <string.h>\n#include <math.h>\n#include <omp.h>\n#include <sys/mman.h>\n#include <errno.h>\n\n/* Student controls: bounded allocations; validation stays outside timed regions. */\nstatic long setting(const char *key,long fallback,long low,long high){\n    const char *v=getenv(key); if(!v) return fallback;\n    char *end; errno=0; long x=strtol(v,&end,10);\n    if(errno || *end || x<low || x>high){fprintf(stderr,"Invalid %s: expected %ld..%ld\\n",key,low,high);exit(2);} return x;\n}\nstatic size_t array_n(size_t fallback){return (size_t)setting("M3_ARRAY_N",(long)fallback,1024,16777216);}\nstatic void require(int ok,const char *name){if(!ok){fprintf(stderr,"VALIDATION_FAIL %s\\n",name);exit(3);}printf("VALIDATION name=%s result=PASS\\n",name);}\nstatic int close_value(double a,double b){return fabs(a-b)<=1e-8*fmax(1.0,fabs(b));}\n\nstatic volatile double g_sink = 0.0;\nstatic double now_sec(void){ return omp_get_wtime(); }\n\nstatic void *xaligned(size_t align, size_t bytes){\n    void *p=NULL;\n    if(posix_memalign(&p,align,bytes)!=0 || !p){\n        fprintf(stderr,"allocation failed: %.1f MiB\\n",bytes/1048576.0);\n        exit(2);\n    }\n    return p;\n}\n\nstatic uint64_t xorshift64(uint64_t *s){\n    uint64_t x=*s;\n    x^=x<<13; x^=x>>7; x^=x<<17; *s=x; return x;\n}\n\nstatic void bench_latency(void){\n    const size_t n=array_n(4u*1024u*1024u);\n    uint32_t *next=(uint32_t*)xaligned(64,n*sizeof(uint32_t));\n    uint32_t *perm=(uint32_t*)xaligned(64,n*sizeof(uint32_t));\n    for(size_t i=0;i<n;++i) perm[i]=(uint32_t)i;\n    uint64_t state=0x123456789abcdefULL;\n    for(size_t i=n-1;i>0;--i){\n        size_t j=(size_t)(xorshift64(&state)%(i+1));\n        uint32_t t=perm[i]; perm[i]=perm[j]; perm[j]=t;\n    }\n    for(size_t i=0;i+1<n;++i) next[perm[i]]=perm[i+1];\n    next[perm[n-1]]=perm[0];\n    volatile uint32_t idx=perm[0];\n    const int laps=4;\n    double t0=now_sec();\n    for(int r=0;r<laps;++r)\n        for(size_t i=0;i<n;++i) idx=next[idx];\n    double t=now_sec()-t0;\n    require(idx==perm[0],"pointer_cycle");\n    double accesses=(double)n*laps;\n    printf("LATENCY n=%zu accesses=%.0f seconds=%.6f ns_per_access=%.2f final=%u\\n",\n           n,accesses,t,t*1e9/accesses,(unsigned)idx);\n    free(next); free(perm);\n}\n\nstatic void init_three(double *a,double *b,double *c,size_t n,int threads){\n    #pragma omp parallel for num_threads(threads) schedule(static)\n    for(size_t i=0;i<n;++i){\n        a[i]=0.0; b[i]=1.0+(double)(i%17)*1e-3; c[i]=2.0+(double)(i%13)*1e-3;\n    }\n}\n\nstatic void bench_stream(void){\n    const size_t n=array_n(8u*1024u*1024u);\n    const int reps=6;\n    double *a=(double*)xaligned(64,n*sizeof(double));\n    double *b=(double*)xaligned(64,n*sizeof(double));\n    double *c=(double*)xaligned(64,n*sizeof(double));\n    int max_threads=omp_get_max_threads();\n    int cand[]={1,2,4,8,16};\n    for(size_t ci=0;ci<sizeof(cand)/sizeof(cand[0]);++ci){\n        int threads=cand[ci]; if(threads>max_threads) continue;\n        init_three(a,b,c,n,threads);\n        double scalar=3.0;\n        #pragma omp parallel for num_threads(threads) schedule(static)\n        for(size_t i=0;i<n;++i) a[i]=b[i]+scalar*c[i];\n        double t0=now_sec();\n        for(int r=0;r<reps;++r){\n            #pragma omp parallel for num_threads(threads) schedule(static)\n            for(size_t i=0;i<n;++i) a[i]=b[i]+scalar*c[i]+1e-12*r;\n        }\n        double t=now_sec()-t0;\n        double bytes=3.0*(double)n*sizeof(double)*reps;\n        double gb=bytes/t/1e9;\n        int ok=1; for(size_t i=0;i<n;++i) if(!close_value(a[i],b[i]+scalar*c[i]+1e-12*(reps-1))) ok=0;\n        require(ok,"stream_values");\n        double check=a[0]+a[n/2]+a[n-1];\n        printf("STREAM threads=%d n=%zu reps=%d seconds=%.6f useful_GB_s=%.2f checksum=%.6f\\n",\n               threads,n,reps,t,gb,check);\n    }\n    free(a); free(b); free(c);\n}\n\nstatic void bench_stride(void){\n    const size_t n=array_n(16u*1024u*1024u);\n    double *a=(double*)xaligned(64,n*sizeof(double));\n    for(size_t i=0;i<n;++i) a[i]=1.0+(double)(i&7)*1e-6;\n    volatile double *v=a;\n    int strides[]={1,2,4,8,16,32,64};\n    const int reps=4;\n    for(size_t si=0;si<sizeof(strides)/sizeof(strides[0]);++si){\n        int stride=strides[si];\n        volatile double sum=0.0;\n        size_t accesses=0;\n        double t0=now_sec();\n        for(int r=0;r<reps;++r){\n            for(size_t i=0;i<n;i+=(size_t)stride){ sum+=v[i]; ++accesses; }\n        }\n        double t=now_sec()-t0;\n        printf("STRIDE stride=%d accesses=%zu seconds=%.6f ns_per_access=%.2f checksum=%.6f\\n",\n               stride,accesses,t,t*1e9/(double)accesses,(double)sum);\n    }\n    free(a);\n}\n\nstatic double sum_row(volatile double *a,int n){\n    double s=0.0;\n    for(int i=0;i<n;++i) for(int j=0;j<n;++j) s+=a[(size_t)i*n+j];\n    return s;\n}\nstatic double sum_col(volatile double *a,int n){\n    double s=0.0;\n    for(int j=0;j<n;++j) for(int i=0;i<n;++i) s+=a[(size_t)i*n+j];\n    return s;\n}\nstatic void bench_matrix(void){\n    const int n=(int)setting("M3_MATRIX_N",2048,16,4096);\n    size_t elems=(size_t)n*n;\n    double *a=(double*)xaligned(64,elems*sizeof(double));\n    for(size_t i=0;i<elems;++i) a[i]=1.0+(double)(i&31)*1e-6;\n    double br=1e99,bc=1e99,sr=0,sc=0;\n    for(int r=0;r<3;++r){\n        double t0=now_sec(); sr=sum_row(a,n); double tr=now_sec()-t0;\n        t0=now_sec(); sc=sum_col(a,n); double tc=now_sec()-t0;\n        if(tr<br) br=tr; if(tc<bc) bc=tc;\n    }\n    require(close_value(sr,sc),"row_column_sum");\n    g_sink+=sr+sc;\n    printf("MATRIX order=row n=%d seconds=%.6f checksum=%.6e\\n",n,br,sr);\n    printf("MATRIX order=column n=%d seconds=%.6f checksum=%.6e\\n",n,bc,sc);\n    printf("MATRIX ratio_column_over_row=%.2f\\n",bc/br);\n    free(a);\n}\n\nstatic void bench_fusion(void){\n    const size_t n=array_n(8u*1024u*1024u);\n    const int reps=5;\n    double *a=(double*)xaligned(64,n*sizeof(double));\n    double *b=(double*)xaligned(64,n*sizeof(double));\n    double *tmp=(double*)xaligned(64,n*sizeof(double));\n    double *out=(double*)xaligned(64,n*sizeof(double));\n    #pragma omp parallel for schedule(static)\n    for(size_t i=0;i<n;++i){ a[i]=1.0+(i&7)*1e-4; b[i]=2.0+(i&15)*1e-4; tmp[i]=out[i]=0.0; }\n    const double scale=0.125;\n    double t0=now_sec();\n    for(int r=0;r<reps;++r){\n        #pragma omp parallel for schedule(static)\n        for(size_t i=0;i<n;++i) tmp[i]=a[i]+b[i];\n        #pragma omp parallel for schedule(static)\n        for(size_t i=0;i<n;++i) out[i]=tmp[i]*scale+1e-12*r;\n    }\n    double two=now_sec()-t0;\n    double c1=out[0]+out[n/2]+out[n-1];\n    int two_ok=1;for(size_t i=0;i<n;++i)if(!close_value(out[i],(a[i]+b[i])*scale+1e-12*(reps-1)))two_ok=0;\n    require(two_ok,"two_pass_values");\n    t0=now_sec();\n    for(int r=0;r<reps;++r){\n        #pragma omp parallel for schedule(static)\n        for(size_t i=0;i<n;++i) out[i]=(a[i]+b[i])*scale+1e-12*r;\n    }\n    double fused=now_sec()-t0;\n    double c2=out[0]+out[n/2]+out[n-1];\n    int ok=1; for(size_t i=0;i<n;++i) if(!close_value(out[i],(a[i]+b[i])*scale+1e-12*(reps-1))) ok=0;\n    require(ok && close_value(c1,c2),"fusion_values");\n    printf("FUSION mode=two_pass n=%zu reps=%d seconds=%.6f checksum=%.6f\\n",n,reps,two,c1);\n    printf("FUSION mode=fused n=%zu reps=%d seconds=%.6f checksum=%.6f\\n",n,reps,fused,c2);\n    printf("FUSION speedup=%.2f\\n",two/fused);\n    free(a); free(b); free(tmp); free(out);\n}\n\nstatic void transpose_naive(double *dst,const double *src,int n){\n    for(int i=0;i<n;++i) for(int j=0;j<n;++j) dst[(size_t)j*n+i]=src[(size_t)i*n+j];\n}\nstatic void transpose_blocked(double *dst,const double *src,int n,int bs){\n    for(int ii=0;ii<n;ii+=bs) for(int jj=0;jj<n;jj+=bs){\n        int imax=ii+bs<n?ii+bs:n, jmax=jj+bs<n?jj+bs:n;\n        for(int i=ii;i<imax;++i) for(int j=jj;j<jmax;++j)\n            dst[(size_t)j*n+i]=src[(size_t)i*n+j];\n    }\n}\nstatic void bench_tiling(void){\n    const int n=(int)setting("M3_MATRIX_N",2048,16,4096);\n    size_t elems=(size_t)n*n;\n    double *src=(double*)xaligned(64,elems*sizeof(double));\n    double *dst=(double*)xaligned(64,elems*sizeof(double));\n    for(size_t i=0;i<elems;++i) src[i]=(double)(i&1023)*1e-3;\n    double t0=now_sec(); transpose_naive(dst,src,n); double tn=now_sec()-t0;\n    double c1=dst[0]+dst[elems/2]+dst[elems-1];\n    int naive_ok=1;for(int i=0;i<n;++i)for(int j=0;j<n;++j)if(dst[(size_t)j*n+i]!=src[(size_t)i*n+j])naive_ok=0;\n    require(naive_ok,"naive_transpose_all_cells");\n    memset(dst,0,elems*sizeof(double));\n    t0=now_sec(); transpose_blocked(dst,src,n,(int)setting("M3_TILE",32,1,512)); double tb=now_sec()-t0;\n    double c2=dst[0]+dst[elems/2]+dst[elems-1];\n    int ok=1; for(int i=0;i<n;++i) for(int j=0;j<n;++j) if(dst[(size_t)j*n+i]!=src[(size_t)i*n+j]) ok=0;\n    require(ok,"transpose_all_cells");\n    printf("TILING workload=transpose mode=naive n=%d seconds=%.6f checksum=%.6f\\n",n,tn,c1);\n    printf("TILING workload=transpose mode=blocked block=%ld n=%d seconds=%.6f checksum=%.6f\\n",setting("M3_TILE",32,1,512),n,tb,c2);\n    printf("TILING speedup=%.2f\\n",tn/tb);\n    free(src); free(dst);\n}\n\ntypedef struct { volatile long long value; } CounterPacked;\ntypedef struct { volatile long long value; char pad[64-sizeof(long long)]; } CounterPadded;\n\nstatic void bench_false_sharing(void){\n    const long long iters=setting("M3_COUNTER_ITERS",2000000,1000,20000000);\n    int max_threads=omp_get_max_threads();\n    int cand[]={1,2,4,8,16};\n    for(size_t ci=0;ci<sizeof(cand)/sizeof(cand[0]);++ci){\n        int threads=cand[ci]; if(threads>max_threads) continue;\n        CounterPacked *p=(CounterPacked*)xaligned(64,(size_t)threads*sizeof(CounterPacked));\n        CounterPadded *q=(CounterPadded*)xaligned(64,(size_t)threads*sizeof(CounterPadded));\n        for(int i=0;i<threads;++i){ p[i].value=0; q[i].value=0; }\n        double t0=now_sec();\n        #pragma omp parallel num_threads(threads)\n        {\n            int tid=omp_get_thread_num();\n            for(long long k=0;k<iters;++k) p[tid].value++;\n        }\n        double tp=now_sec()-t0;\n        long long sp=0; for(int i=0;i<threads;++i) sp+=p[i].value;\n        t0=now_sec();\n        #pragma omp parallel num_threads(threads)\n        {\n            int tid=omp_get_thread_num();\n            for(long long k=0;k<iters;++k) q[tid].value++;\n        }\n        double tq=now_sec()-t0;\n        long long sq=0; for(int i=0;i<threads;++i) sq+=q[i].value;\n        require(sp==iters*threads && sq==iters*threads,"private_counters");\n        printf("FALSE_SHARING threads=%d layout=packed seconds=%.6f total=%lld\\n",threads,tp,sp);\n        printf("FALSE_SHARING threads=%d layout=padded seconds=%.6f total=%lld\\n",threads,tq,sq);\n        printf("FALSE_SHARING threads=%d packed_over_padded=%.2f\\n",threads,tp/tq);\n        free(p); free(q);\n    }\n}\n\ntypedef struct {\n    double x,y,z,vx,vy,vz,mass;\n    uint32_t id,pad;\n} Particle;\n\nstatic void bench_aossoa(void){\n    const size_t n=array_n(4u*1024u*1024u);\n    const int reps=4;\n    Particle *p=(Particle*)xaligned(64,n*sizeof(Particle));\n    double *x=(double*)xaligned(64,n*sizeof(double));\n    double *y=(double*)xaligned(64,n*sizeof(double));\n    double *z=(double*)xaligned(64,n*sizeof(double));\n    double *vx=(double*)xaligned(64,n*sizeof(double));\n    double *vy=(double*)xaligned(64,n*sizeof(double));\n    double *vz=(double*)xaligned(64,n*sizeof(double));\n    double *mass=(double*)xaligned(64,n*sizeof(double));\n    #pragma omp parallel for schedule(static)\n    for(size_t i=0;i<n;++i){\n        double f=(double)(i&1023)*1e-6;\n        p[i]=(Particle){1+f,2+f,3+f,0.1,0.2,0.3,1.0+(i&7),(uint32_t)i,0};\n        x[i]=p[i].x; y[i]=p[i].y; z[i]=p[i].z;\n        vx[i]=p[i].vx; vy[i]=p[i].vy; vz[i]=p[i].vz; mass[i]=p[i].mass;\n    }\n    double t0=now_sec();\n    for(int r=0;r<reps;++r){\n        #pragma omp parallel for schedule(static)\n        for(size_t i=0;i<n;++i) p[i].x+=1e-6;\n    }\n    double ta=now_sec()-t0;\n    double ca=p[0].x+p[n/2].x+p[n-1].x;\n    t0=now_sec();\n    for(int r=0;r<reps;++r){\n        #pragma omp parallel for schedule(static)\n        for(size_t i=0;i<n;++i) x[i]+=1e-6;\n    }\n    double ts=now_sec()-t0;\n    double cs=x[0]+x[n/2]+x[n-1];\n    printf("AOSSOA kernel=x_only layout=AoS n=%zu reps=%d seconds=%.6f checksum=%.6f struct_bytes=%zu\\n",n,reps,ta,ca,sizeof(Particle));\n    printf("AOSSOA kernel=x_only layout=SoA n=%zu reps=%d seconds=%.6f checksum=%.6f field_bytes=%zu\\n",n,reps,ts,cs,sizeof(double));\n    printf("AOSSOA kernel=x_only aos_over_soa=%.2f\\n",ta/ts);\n\n    const double dt=0.001;\n    t0=now_sec();\n    for(int r=0;r<reps;++r){\n        #pragma omp parallel for schedule(static)\n        for(size_t i=0;i<n;++i){\n            p[i].x+=p[i].vx*dt; p[i].y+=p[i].vy*dt; p[i].z+=p[i].vz*dt;\n            p[i].vx+=p[i].mass*1e-9;\n        }\n    }\n    double taf=now_sec()-t0;\n    ca=p[0].x+p[n/2].y+p[n-1].z+p[n/3].vx;\n    t0=now_sec();\n    for(int r=0;r<reps;++r){\n        #pragma omp parallel for schedule(static)\n        for(size_t i=0;i<n;++i){\n            x[i]+=vx[i]*dt; y[i]+=vy[i]*dt; z[i]+=vz[i]*dt;\n            vx[i]+=mass[i]*1e-9;\n        }\n    }\n    double tsf=now_sec()-t0;\n    cs=x[0]+y[n/2]+z[n-1]+vx[n/3];\n    printf("AOSSOA kernel=multi_field layout=AoS n=%zu reps=%d seconds=%.6f checksum=%.6f\\n",n,reps,taf,ca);\n    printf("AOSSOA kernel=multi_field layout=SoA n=%zu reps=%d seconds=%.6f checksum=%.6f\\n",n,reps,tsf,cs);\n    printf("AOSSOA kernel=multi_field aos_over_soa=%.2f\\n",taf/tsf);\n    int ok=1; for(size_t i=0;i<n;++i) if(!close_value(p[i].x,x[i])||!close_value(p[i].y,y[i])||!close_value(p[i].z,z[i])||!close_value(p[i].vx,vx[i])) ok=0;\n    require(ok,"particle_layout_equivalence");\n    free(p); free(x); free(y); free(z); free(vx); free(vy); free(vz); free(mass);\n}\n\nstatic double sum_parallel(const double *a,size_t n){\n    double s=0.0;\n    #pragma omp parallel for reduction(+:s) schedule(static)\n    for(size_t i=0;i<n;++i) s+=a[i];\n    return s;\n}\nstatic void bench_first_touch(void){\n    const size_t n=array_n(32u*1024u*1024u);\n    const int reps=4;\n    double *a=mmap(NULL,n*sizeof(double),PROT_READ|PROT_WRITE,MAP_PRIVATE|MAP_ANONYMOUS,-1,0);\n    if(a==MAP_FAILED){perror("mmap");exit(2);}\n    for(size_t i=0;i<n;++i) a[i]=1.0;\n    double t0=now_sec(); double s1=0.0;\n    for(int r=0;r<reps;++r) s1+=sum_parallel(a,n);\n    double serial_touch=now_sec()-t0;\n    munmap(a,n*sizeof(double));\n\n    a=mmap(NULL,n*sizeof(double),PROT_READ|PROT_WRITE,MAP_PRIVATE|MAP_ANONYMOUS,-1,0);\n    if(a==MAP_FAILED){perror("mmap");exit(2);}\n    #pragma omp parallel for schedule(static)\n    for(size_t i=0;i<n;++i) a[i]=1.0;\n    t0=now_sec(); double s2=0.0;\n    for(int r=0;r<reps;++r) s2+=sum_parallel(a,n);\n    double parallel_touch=now_sec()-t0;\n    require(close_value(s1,(double)n*reps) && close_value(s2,s1),"first_touch_sums");\n    printf("FIRST_TOUCH init=serial n=%zu reps=%d seconds=%.6f checksum=%.6e\\n",n,reps,serial_touch,s1);\n    printf("FIRST_TOUCH init=parallel n=%zu reps=%d seconds=%.6f checksum=%.6e\\n",n,reps,parallel_touch,s2);\n    printf("FIRST_TOUCH serial_over_parallel=%.2f\\n",serial_touch/parallel_touch);\n    munmap(a,n*sizeof(double));\n}\n\nstatic void bench_prefetch(void){\n    const size_t n=array_n(16u*1024u*1024u);\n    const int reps=3;\n    double *a=(double*)xaligned(64,n*sizeof(double));\n    for(size_t i=0;i<n;++i) a[i]=1.0+(i&15)*1e-6;\n    volatile double sum=0.0;\n    double t0=now_sec();\n    for(int r=0;r<reps;++r) for(size_t i=0;i<n;++i) sum+=a[i];\n    double tn=now_sec()-t0; double c1=sum;\n    sum=0.0; const size_t d=(size_t)setting("M3_PREFETCH_DISTANCE",64,1,4096);\n    t0=now_sec();\n    for(int r=0;r<reps;++r) for(size_t i=0;i<n;++i){\n        if(i+d<n) __builtin_prefetch(&a[i+d],0,1);\n        sum+=a[i];\n    }\n    double tp=now_sec()-t0; double c2=sum;\n    require(close_value(c1,c2),"prefetch_sum");\n    printf("PREFETCH mode=none n=%zu reps=%d seconds=%.6f checksum=%.6f\\n",n,reps,tn,c1);\n    printf("PREFETCH mode=software distance=%zu n=%zu reps=%d seconds=%.6f checksum=%.6f\\n",d,n,reps,tp,c2);\n    printf("PREFETCH speedup_none_over_software=%.2f\\n",tn/tp);\n    free(a);\n}\n\nstatic void bench_sparse(void){\n    const int n=(int)setting("M3_SPARSE_N",200000,16,1000000), per=(int)setting("M3_NNZ_PER_ROW",5,1,32), nnz=n*per;\n    double *val=(double*)xaligned(64,(size_t)nnz*sizeof(double));\n    int *col=(int*)xaligned(64,(size_t)nnz*sizeof(int));\n    int *row=(int*)xaligned(64,(size_t)(n+1)*sizeof(int));\n    double *x=(double*)xaligned(64,(size_t)n*sizeof(double));\n    double *y=(double*)xaligned(64,(size_t)n*sizeof(double));\n    #pragma omp parallel for schedule(static)\n    for(int i=0;i<n;++i){\n        x[i]=1.0+(i&31)*1e-6;\n        row[i]=i*per;\n        for(int k=0;k<per;++k){\n            int p=i*per+k;\n            col[p]=(int)(((uint64_t)i*1315423911ULL+(uint64_t)(k+1)*2654435761ULL)%(uint64_t)n);\n            val[p]=1.0/(double)(k+1);\n        }\n    }\n    row[n]=nnz;\n    double t0=now_sec();\n    #pragma omp parallel for schedule(static)\n    for(int i=0;i<n;++i){\n        double s=0.0;\n        for(int j=row[i];j<row[i+1];++j) s+=val[j]*x[col[j]];\n        y[i]=s;\n    }\n    double t=now_sec()-t0;\n    int ok=1; for(int i=0;i<n;++i){double ref=0;for(int j=row[i];j<row[i+1];++j)ref+=val[j]*x[col[j]];if(!close_value(y[i],ref))ok=0;}\n    require(ok,"csr_all_rows");\n    double csr=(double)nnz*(sizeof(double)+sizeof(int))+(double)(n+1)*sizeof(int);\n    double dense=(double)n*(double)n*sizeof(double);\n    double check=y[0]+y[n/2]+y[n-1];\n    printf("SPARSE n=%d nnz=%d nnz_per_row=%d csr_MiB=%.2f dense_TiB=%.2f seconds=%.6f checksum=%.6f\\n",\n           n,nnz,per,csr/1048576.0,dense/1099511627776.0,t,check);\n    free(val); free(col); free(row); free(x); free(y);\n}\n\n\nstatic double add_plain(const double *a,const double *b,double *c,size_t n){\n    double t0=now_sec();\n    for(size_t i=0;i<n;++i) c[i]=a[i]+b[i];\n    return now_sec()-t0;\n}\nstatic double add_unrolled4(const double *a,const double *b,double *c,size_t n){\n    double t0=now_sec();\n    size_t i=0;\n    for(;i+3<n;i+=4){\n        c[i]=a[i]+b[i];\n        c[i+1]=a[i+1]+b[i+1];\n        c[i+2]=a[i+2]+b[i+2];\n        c[i+3]=a[i+3]+b[i+3];\n    }\n    for(;i<n;++i) c[i]=a[i]+b[i];\n    return now_sec()-t0;\n}\nstatic void bench_unroll(void){\n    const size_t n=array_n(16u*1024u*1024u);\n    const int reps=5;\n    double *a=(double*)xaligned(64,n*sizeof(double));\n    double *b=(double*)xaligned(64,n*sizeof(double));\n    double *out=(double*)xaligned(64,n*sizeof(double));\n    for(size_t i=0;i<n;++i){ a[i]=1.0+(i&7)*1e-6; b[i]=2.0+(i&15)*1e-6; }\n    double plain=0.0,unrolled=0.0;\n    for(int r=0;r<reps;++r) plain+=add_plain(a,b,out,n);\n    double c1=out[0]+out[n/2]+out[n-1];\n    int plain_ok=1;for(size_t i=0;i<n;++i)if(out[i]!=a[i]+b[i])plain_ok=0;\n    require(plain_ok,"plain_vector_add");\n    for(int r=0;r<reps;++r) unrolled+=add_unrolled4(a,b,out,n);\n    double c2=out[0]+out[n/2]+out[n-1];\n    int ok=1;for(size_t i=0;i<n;++i)if(out[i]!=a[i]+b[i])ok=0;\n    require(ok && close_value(c1,c2),"unroll_with_tail");\n    printf("UNROLL mode=plain n=%zu reps=%d seconds=%.6f checksum=%.6f\\n",n,reps,plain,c1);\n    printf("UNROLL mode=unrolled4 n=%zu reps=%d seconds=%.6f checksum=%.6f\\n",n,reps,unrolled,c2);\n    printf("UNROLL plain_over_unrolled=%.2f\\n",plain/unrolled);\n    free(a); free(b); free(out);\n}\n\nstatic void usage(const char *p){\n    fprintf(stderr,"usage: %s latency|stream|stride|matrix|false_sharing|aossoa|tiling|fusion|first_touch|prefetch|sparse|unroll|all\\n",p);\n}\nint main(int argc,char **argv){\n    if(argc<2){ usage(argv[0]); return 1; }\n    omp_set_dynamic(0);\n    omp_set_num_threads((int)setting("M3_THREADS",omp_get_max_threads(),1,16));\n    const char *m=argv[1];\n    if(!strcmp(m,"latency")) bench_latency();\n    else if(!strcmp(m,"stream")) bench_stream();\n    else if(!strcmp(m,"stride")) bench_stride();\n    else if(!strcmp(m,"matrix")) bench_matrix();\n    else if(!strcmp(m,"false_sharing")) bench_false_sharing();\n    else if(!strcmp(m,"aossoa")) bench_aossoa();\n    else if(!strcmp(m,"tiling")) bench_tiling();\n    else if(!strcmp(m,"fusion")) bench_fusion();\n    else if(!strcmp(m,"first_touch")) bench_first_touch();\n    else if(!strcmp(m,"prefetch")) bench_prefetch();\n    else if(!strcmp(m,"sparse")) bench_sparse();\n    else if(!strcmp(m,"unroll")) bench_unroll();\n    else if(!strcmp(m,"all")){\n        bench_latency(); bench_stream(); bench_stride(); bench_matrix();\n        bench_false_sharing(); bench_aossoa(); bench_tiling(); bench_fusion();\n        bench_first_touch(); bench_prefetch(); bench_sparse(); bench_unroll();\n    } else { usage(argv[0]); return 1; }\n    fprintf(stderr,"sink=%f\\n",g_sink);\n    return 0;\n}\n', 'm3s2_memory_lab.sbatch': '#!/bin/bash\n#SBATCH --job-name=m3s2_memory\n#SBATCH --partition=cpu\n#SBATCH --nodes=1\n#SBATCH --ntasks=1\n#SBATCH --cpus-per-task=16\n#SBATCH --time=00:08:00\n#SBATCH --mem=4G\n#SBATCH --output=m3s2_memory-%j.out\n\nset -euo pipefail\nif command -v module >/dev/null 2>&1; then module purge; module load foss/2023b; fi\nexport OMP_NUM_THREADS=${M3_THREADS:-${SLURM_CPUS_PER_TASK:-16}}\nexport OMP_DYNAMIC=FALSE\nexport OMP_PROC_BIND=spread\nexport OMP_PLACES=cores\n\necho "M3S2_MEMORY_LAB_START job=${SLURM_JOB_ID:-none} host=$(hostname) cpus=${SLURM_CPUS_PER_TASK:-unknown}"\necho "===== TOPOLOGY ====="\nlscpu | grep -E \'Architecture|Model name|Socket|Core|Thread|NUMA|L1d|L2|L3\' || true\necho "CACHE_LINE_BYTES=$(getconf LEVEL1_DCACHE_LINESIZE 2>/dev/null || echo unknown)"\nif command -v numactl >/dev/null 2>&1; then numactl --hardware || true; fi\necho "===== END TOPOLOGY ====="\n\ngcc -O3 -march=native -fopenmp -std=c11 m3s2_memory_lab.c -o m3s2_memory_lab -lm -lm\nrm -f vectorization.log\ngcc -O3 -march=native -fopt-info-vec-all=vectorization.log -c m3s2_vectorization.c -o m3s2_vectorization.o || true\n\necho "===== VECTOR REPORT ====="\nif [ -f vectorization.log ]; then\n    grep -E \'vectorized|missed|not vectorized|depend\' vectorization.log | head -n 40 || cat vectorization.log | head -n 40\nelse\n    echo "No GCC vectorization report produced."\nfi\necho "===== END VECTOR REPORT ====="\n\nrun_exp(){\n  label="$1"; mode="$2"\n  echo "===== ${label} ====="\n  ./m3s2_memory_lab "$mode"\n  echo "===== END ${label} ====="\n}\n\nrun_exp "EXPERIMENT 1 LATENCY" latency\nrun_exp "EXPERIMENT 2 BANDWIDTH" stream\nrun_exp "EXPERIMENT 3 CACHE LINE STRIDE" stride\nrun_exp "EXPERIMENT 4 ROW VS COLUMN" matrix\nrun_exp "EXPERIMENT 5 FALSE SHARING" false_sharing\nrun_exp "EXPERIMENT 6 AOS VS SOA" aossoa\nrun_exp "EXPERIMENT 7 BLOCKING" tiling\nrun_exp "EXPERIMENT 8 LOOP FUSION" fusion\nrun_exp "EXPERIMENT 9 FIRST TOUCH" first_touch\nrun_exp "EXPERIMENT 10 PREFETCH" prefetch\nrun_exp "EXPERIMENT 11 SPARSE CSR" sparse\nrun_exp "EXPERIMENT 12 LOOP UNROLLING" unroll\n\necho "M3S2_MEMORY_LAB_RESULT=PASS"\n', 'm3s2_vectorization.c': '\n#include <stddef.h>\n\n__attribute__((noinline))\nvoid vectorizable_saxpy(size_t n, double * restrict y, const double * restrict x, double a){\n    for(size_t i=0;i<n;++i) y[i] = a*x[i] + y[i];\n}\n\n__attribute__((noinline))\nvoid loop_carried_dependency(size_t n, double *x, const double *a){\n    for(size_t i=1;i<n;++i) x[i] = x[i-1] + a[i];\n}\n'}
for name,content in DEMO_SOURCES.items():
    (DEMO_DIR/name).write_text(content)
print("Prepared:", DEMO_DIR)


# 2 — REAL CLUSTER EXPERIMENT: submit once, measure the memory system

The job compiles the C/OpenMP kernels **inside the allocation** and then runs all experiments.

### Predict before running

1. Will memory bandwidth keep doubling from 1 → 2 → 4 → 8 → 16 threads?
2. Will a stride of 8 doubles behave like stride 1?
3. Will row-major and column-major traversal take the same time?
4. Can threads slow each other down even when they write different array elements?
5. If a kernel touches only one field of a large structure, will layout matter?

> Run the next cell once. The job may take a minute or two depending on queue state and node speed.


In [ ]:
def clean_slurm_env():
    env=os.environ.copy()
    for key in ('SLURM_MEM_PER_CPU','SLURM_MEM_PER_GPU','SLURM_MEM_PER_NODE'):
        env.pop(key,None)
    return env


def read_output(job_id):
    if not job_id: return ''
    p=DEMO_DIR / f'm3s2_memory-{job_id}.out'
    if not p.exists():
        print('Output file not found yet:',p)
        return ''
    return p.read_text(errors='replace')


def kv_lines(text,prefix):
    rows=[]
    for line in text.splitlines():
        if not line.startswith(prefix): continue
        row={}
        for token in line.split():
            if '=' in token:
                k,v=token.split('=',1); row[k]=v
        rows.append(row)
    return rows


def section(text,label):
    a=f'===== {label} ====='; b=f'===== END {label} ====='
    i=text.find(a); j=text.find(b)
    if i<0 or j<0: return ''
    return text[i+len(a):j].strip()


def show_rows(headers, rows):
    if not rows:
        print('No rows loaded.')
        return
    widths=[len(str(h)) for h in headers]
    txt=[[str(v) for v in r] for r in rows]
    for r in txt:
        widths=[max(w,len(v)) for w,v in zip(widths,r)]
    print(' | '.join(str(h).ljust(w) for h,w in zip(headers,widths)))
    print('-+-'.join('-'*w for w in widths))
    for r in txt:
        print(' | '.join(v.ljust(w) for v,w in zip(r,widths)))


def collect_memory_results(job_id, timeout=600, poll=10):
    start=time.monotonic();last=None
    required=["LATENCY","STREAM","STRIDE","MATRIX","FALSE_SHARING","AOSSOA","TILING","FUSION","FIRST_TOUCH","PREFETCH","SPARSE","UNROLL"]
    while True:
        path=DEMO_DIR/f"m3s2_memory-{job_id}.out"
        output=path.read_text(errors="replace") if path.exists() else ""
        if "M3S2_MEMORY_LAB_RESULT=PASS" in output:
            if any(not kv_lines(output,key) for key in required): raise RuntimeError("PASS marker but missing experiment records")
            return output
        q=subprocess.run(["squeue","-h","-j",str(job_id),"-o","%T %R"],capture_output=True,text=True,timeout=15)
        if q.returncode: raise RuntimeError("Cannot check queue: "+q.stderr)
        status=q.stdout.strip()
        if not status and shutil.which("sacct"):
            a=subprocess.run(["sacct","-n","-X","-j",str(job_id),"--format=State,ExitCode","--parsable2"],capture_output=True,text=True,timeout=15)
            if a.returncode==0: status=a.stdout.strip()
            if any(word in status for word in ["FAILED","CANCELLED","TIMEOUT","OUT_OF_MEMORY","NODE_FAIL","PREEMPTED","BOOT_FAIL"]):
                print(output[-5000:]);raise RuntimeError("Job failed: "+status+". Fix the issue, then use REPEAT_SAME_EXPERIMENT=True.")
        elapsed=time.monotonic()-start;message=(status,int(elapsed)//30)
        if message!=last: print(f"Job {job_id}: {status or 'waiting for output'}; {int(elapsed)} seconds",flush=True);last=message
        if elapsed>=timeout: raise RuntimeError(f"Job {job_id} has no complete output yet. Rerun this SAME cell later to reuse the job, then Run All Below. Diagnose: sacct -j {job_id} --format=JobID,State,ExitCode")
        time.sleep(min(poll,timeout-elapsed))

# Plot without Matplotlib: Jupyter displays standard-library SVG.
def plot_values(labels,values,title,ylabel):
    from html import escape
    from IPython.display import SVG,display
    if not labels or len(labels)!=len(values): raise ValueError("Matching, nonempty plot data required")
    ymax=max(values) or 1;parts=['<svg xmlns="http://www.w3.org/2000/svg" width="780" height="350" viewBox="0 0 780 350"><rect width="780" height="350" fill="white"/>',f'<title>{escape(title)}</title>',f'<text x="70" y="25" font-size="17">{escape(title)}</text>']
    for i,(label,value) in enumerate(zip(labels,values)):
        x=75+i*620/len(labels);h=230*value/ymax
        parts.extend([f'<rect x="{x}" y="{280-h}" width="{450/len(labels)}" height="{h}" fill="#2563eb"/>',f'<text x="{x}" y="300" font-size="11">{escape(str(label))}</text>',f'<text x="{x}" y="{270-h}" font-size="11">{value:.3g}</text>'])
    parts.append(f'<text x="70" y="330" font-size="12">{escape(ylabel)}</text></svg>');display(SVG("".join(parts)))

# The actual C function can be inspected beside each analysis.
def show_kernel(name):
    text=(DEMO_DIR/"m3s2_memory_lab.c").read_text();start=text.index("static void "+name+"(");brace=text.index("{",start);level=1;end=brace+1
    while level:
        if text[end]=="{": level+=1
        elif text[end]=="}": level-=1
        end+=1
    print(text[start:end])


In [ ]:
# Run All submits once per configuration, then waits for real output.
import json, hashlib
RUN_CLUSTER=True
REPEAT_SAME_EXPERIMENT=False  # True only for an intentional repeat (reset afterwards).
state_path=DEMO_DIR/"last_job.json"
saved=json.loads(state_path.read_text()) if state_path.exists() else {}
fingerprint=hashlib.sha256(json.dumps({"settings":LAB_SETTINGS,"sources":DEMO_SOURCES},sort_keys=True).encode()).hexdigest()
M3S2_OUTPUT=""
M3S2_JOB_ID=saved.get("job_id")
if RUN_CLUSTER and shutil.which("sbatch"):
    same=saved.get("fingerprint")==fingerprint
    if M3S2_JOB_ID and (not same or REPEAT_SAME_EXPERIMENT):
        q=subprocess.run(["squeue","-h","-j",str(M3S2_JOB_ID)],capture_output=True,text=True,timeout=15)
        if q.returncode: raise RuntimeError(q.stderr)
        if q.stdout.strip(): raise RuntimeError(f"Job {M3S2_JOB_ID} is active. Collect it before changing settings.")
        M3S2_JOB_ID=None
    if not M3S2_JOB_ID:
        env=clean_slurm_env();env.update({k:str(v) for k,v in LAB_SETTINGS.items()})
        result=subprocess.run(["sbatch","--parsable","--export=ALL",str(DEMO_DIR/"m3s2_memory_lab.sbatch")],cwd=DEMO_DIR,env=env,capture_output=True,text=True,timeout=30)
        if result.returncode: raise RuntimeError("Submission failed: "+result.stderr)
        M3S2_JOB_ID=result.stdout.strip().split(";")[0]
        if not str(M3S2_JOB_ID).isdigit(): raise RuntimeError("Unexpected job ID: "+str(M3S2_JOB_ID))
        state_path.write_text(json.dumps({"job_id":M3S2_JOB_ID,"fingerprint":fingerprint,"settings":LAB_SETTINGS},indent=2))
        print("Submitted and saved job",M3S2_JOB_ID)
    else: print("Reusing job",M3S2_JOB_ID,"with unchanged settings. No duplicate submitted.")
    M3S2_OUTPUT=collect_memory_results(M3S2_JOB_ID)
    (DEMO_DIR/f"experiment-{M3S2_JOB_ID}.json").write_text(json.dumps({"job_id":M3S2_JOB_ID,"settings":LAB_SETTINGS,"output":M3S2_OUTPUT},indent=2))
    print("RESULTS READY: analyses below use your measured data.")
    print(section(M3S2_OUTPUT,"TOPOLOGY"))
else:
    print("MODEL MODE: no Slurm job. Small correctness activities and models work; no real timing claim.")


# 3 — Latency and bandwidth are different limits

### Mental model

- **Latency:** how long until a requested value arrives.
- **Bandwidth:** how much data can be moved per second once transfer is flowing.

The lab measures both:

- a dependent, randomized pointer/index chase → **latency-sensitive**;
- an OpenMP streaming triad → **bandwidth-sensitive**.

### Predict

Will adding threads to the streaming kernel eventually stop increasing bandwidth?


In [ ]:
lat=kv_lines(M3S2_OUTPUT,'LATENCY')
stream=kv_lines(M3S2_OUTPUT,'STREAM')

if lat:
    print('Dependent random-access latency:')
    print(f"  {float(lat[0]['ns_per_access']):.2f} ns/access")
    print()

if stream:
    rows=[]
    for r in stream:
        rows.append((r['threads'],f"{float(r['seconds']):.4f}",f"{float(r['useful_GB_s']):.2f}"))
    show_rows(['Threads','Time (s)','Useful GB/s*'],rows)
    print('\n* Useful bytes / time; real DRAM traffic can be higher because of cache/write policies.')
else:
    print('Run Section 2 on the cluster first.')

if stream: plot_values([r["threads"] for r in stream],[float(r["useful_GB_s"]) for r in stream],"Measured streaming throughput","Useful GB/s")


### Observe and explain

The two measurements expose different limits:

- dependent random access cannot easily overlap the next access until the current address is known;
- streaming is predictable and can exploit prefetching, multiple memory requests and many memory channels.

For the thread-scaling table, identify the first point where additional threads produce little extra GB/s.

> **A memory-bound loop can stop scaling even when more CPU cores are available.**


### Change, measure, explain

Change M3_THREADS from 16 to 4. Compare the best useful GB/s, not just the number of workers. The pointer chase working set may fit in last-level cache: ns/access is not a pure DRAM latency measurement.

**Your answer:** I changed ___; the measured result changed from ___ to ___; my explanation is ___; an alternative explanation is ___.


In [ ]:
SHOW_C_CODE=True  # Set True to see the actual measured function, including OpenMP directives.
if SHOW_C_CODE: show_kernel('bench_stream')


# 4 — Cache lines: stride changes how much of each transfer is useful

Assume a typical **64-byte cache line** and 8-byte doubles.

```text
stride 1:  A[0] A[1] A[2] A[3] A[4] A[5] A[6] A[7]
           └──────────── one cache line ────────────┘

stride 8:  A[0] ....... A[8] ....... A[16] .......
           roughly one useful double per cache line
```

### Predict

Which stride should give the lowest time per useful access?


In [ ]:
rows=kv_lines(M3S2_OUTPUT,'STRIDE')
if rows:
    table=[]
    for r in rows:
        table.append((r['stride'],r['accesses'],f"{float(r['ns_per_access']):.2f}"))
    show_rows(['Stride (doubles)','Accesses','ns / useful access'],table)
else:
    print('Run Section 2 first.')

if rows: plot_values([r["stride"] for r in rows],[float(r["ns_per_access"]) for r in rows],"Stride: cost per useful access","ns/access")


### Explain

The algorithm is still “sum selected values”, but the **usefulness of each fetched cache line changes**.

At stride 1, adjacent values can be consumed from the same line. At larger strides, fewer bytes from each transferred line may contribute useful work.

This is **spatial locality** made visible.

The exact curve is hardware-dependent because modern CPUs also prefetch and overlap memory requests. Focus on the trend, not one magic stride value.


### Change, measure, explain

Change M3_ARRAY_N to 1048576. Compare ns/access; strides perform different numbers of accesses. Volatile accesses and a scalar accumulator deliberately constrain optimisation, so this is not peak memory bandwidth.

**Your answer:** I changed ___; the measured result changed from ___ to ___; my explanation is ___; an alternative explanation is ___.


In [ ]:
SHOW_C_CODE=True  # Set True to see the actual measured function, including OpenMP directives.
if SHOW_C_CODE: show_kernel('bench_stride')


# 5 — Same matrix, same arithmetic, different traversal

C-style arrays are stored in **row-major** order.

```c
// Row traversal
for (i = 0; i < N; ++i)
    for (j = 0; j < N; ++j)
        sum += A[i][j];

// Column traversal
for (j = 0; j < N; ++j)
    for (i = 0; i < N; ++i)
        sum += A[i][j];
```

Same `N²` additions. Same values. Same processor.

### Predict

Which traversal better matches physical layout?


In [ ]:
rows=kv_lines(M3S2_OUTPUT,'MATRIX')
if rows:
    table=[]
    for r in rows:
        if 'order' in r:
            table.append((r['order'],r['n'],f"{float(r['seconds']):.4f}",r['checksum']))
    show_rows(['Order','N','Time (s)','Checksum'],table)
    ratio=[r for r in rows if 'ratio_column_over_row' in r]
    if ratio:
        print(f"\nColumn / row runtime ratio: {float(ratio[0]['ratio_column_over_row']):.2f}x")
else:
    print('Run Section 2 first.')

if rows: plot_values([r["order"] for r in rows if "order" in r],[float(r["seconds"]) for r in rows if "order" in r],"Row vs column traversal","seconds")


### Explain

The key lesson is not “rows are always faster”.

It is:

> **Traversal should match the contiguous dimension of the actual layout.**

In C row-major storage, changing the inner loop from `j` to `i` turns nearby accesses into large-stride accesses.

Big-O remains `O(N²)` in both cases. Big-O does **not** describe the complete memory cost.


### Change, measure, explain

Change M3_MATRIX_N to 1024 or 2049. Both traversals sum the same matrix. Matching sums are necessary; speed alone does not establish correctness.

**Your answer:** I changed ___; the measured result changed from ___ to ___; my explanation is ___; an alternative explanation is ___.


In [ ]:
SHOW_C_CODE=True  # Set True to see the actual measured function, including OpenMP directives.
if SHOW_C_CODE: show_kernel('bench_matrix')


# 6 — False sharing: different variables can still share one cache line

This experiment is deliberately simple.

Each OpenMP thread increments **its own counter**. There is no logical data race.

Two layouts are compared:

```text
PACKED
[T0 counter][T1 counter][T2 counter][T3 counter] ...
          several counters can occupy one cache line

PADDED
[T0 counter + padding] [T1 counter + padding] ...
          each counter starts in its own 64-byte slot
```

### Predict

If threads write different counters, why could packed storage still be slower?


In [ ]:
rows=kv_lines(M3S2_OUTPUT,'FALSE_SHARING')
if rows:
    by={}
    ratios={}
    for r in rows:
        t=int(r['threads'])
        if 'layout' in r:
            by.setdefault(t,{})[r['layout']]=float(r['seconds'])
        if 'packed_over_padded' in r:
            ratios[t]=float(r['packed_over_padded'])
    table=[]
    for t in sorted(by):
        table.append((t,f"{by[t].get('packed',float('nan')):.4f}",f"{by[t].get('padded',float('nan')):.4f}",f"{ratios.get(t,float('nan')):.2f}x"))
    show_rows(['Threads','Packed (s)','Padded (s)','Packed/Padded'],table)
else:
    print('Run Section 2 first.')


### Explain

Caches maintain coherence at **cache-line granularity**, not C-variable granularity.

If multiple writable counters sit on the same line, cores can repeatedly invalidate each other's copies even though they never update the same counter.

> **No data race does not imply no cache-coherence traffic.**

This is why data layout and thread ownership can matter even for apparently independent work.


### Change, measure, explain

Change M3_THREADS to 1, then 8. The padding assumes a 64-byte coherence line; check topology. Volatile forces accesses here but is not a general thread-safety mechanism.

**Your answer:** I changed ___; the measured result changed from ___ to ___; my explanation is ___; an alternative explanation is ___.


In [ ]:
SHOW_C_CODE=True  # Set True to see the actual measured function, including OpenMP directives.
if SHOW_C_CODE: show_kernel('bench_false_sharing')


# 7 — Data layout: Array of Structures vs Structure of Arrays

Consider particles with:

```text
{x, y, z, vx, vy, vz, mass, id}
```

The first measured kernel updates **only `x`**.

### AoS

```text
[x y z vx vy vz mass id] [x y z vx vy vz mass id] ...
```

### SoA

```text
x:  [x x x x x ...]
y:  [y y y y y ...]
z:  [z z z z z ...]
...
```

### Predict

Which layout moves less irrelevant data for an `x`-only kernel?


In [ ]:
rows=kv_lines(M3S2_OUTPUT,'AOSSOA')
if rows:
    table=[]
    for r in rows:
        if 'layout' in r:
            table.append((r['kernel'],r['layout'],r['n'],f"{float(r['seconds']):.4f}"))
    show_rows(['Kernel','Layout','Particles','Time (s)'],table)
    print()
    for r in rows:
        if 'aos_over_soa' in r:
            print(f"{r['kernel']}: AoS/SoA runtime = {float(r['aos_over_soa']):.2f}x")
else:
    print('Run Section 2 first.')


### Explain

For `x`-only work, SoA usually lets cache lines contain far more useful values.

But the notebook also runs a **multi-field** particle update. Compare both results before making a universal claim.

> **AoS vs SoA has no universal winner. Choose the representation that matches the hot access pattern.**

This is data-oriented thinking: design around how the program actually consumes data.


### Change, measure, explain

Change M3_ARRAY_N to 1048576. Compare x-only and multi-field results; explain why their layout advantages can differ.

**Your answer:** I changed ___; the measured result changed from ___ to ___; my explanation is ___; an alternative explanation is ___.


In [ ]:
SHOW_C_CODE=True  # Set True to see the actual measured function, including OpenMP directives.
if SHOW_C_CODE: show_kernel('bench_aossoa')


# 8 — Qualitative Roofline: arithmetic intensity changes what can limit performance

Roofline reasoning starts from one question:

> **How much useful arithmetic is performed for each byte moved?**

Arithmetic intensity:

\[
AI = \frac{\text{floating-point operations}}{\text{bytes moved}}
\]

We will use the **measured streaming bandwidth** from this node, but an **illustrative compute ceiling**. This is a MODEL, not a full Roofline measurement.

### Predict

Which should be more sensitive to memory bandwidth?

- vector add: very few operations per byte;
- a small stencil: more reuse, but still substantial memory traffic;
- blocked dense matrix multiplication: high reuse and many operations per loaded block.


In [ ]:
stream=kv_lines(M3S2_OUTPUT,'STREAM')
if stream:
    measured_bw=max(float(r['useful_GB_s']) for r in stream)
else:
    measured_bw=50.0
    print('No cluster bandwidth loaded; using 50 GB/s only for the MODEL.')

MODEL_COMPUTE_CEILING_GFLOPS=200.0  # deliberately illustrative
kernels=[
    ('Vector add', 1.0/24.0),
    ('5-point stencil (simple model)', 0.20),
    ('Blocked dense matmul (illustrative)', 8.0),
]

rows=[]
for name,ai in kernels:
    memory_roof=measured_bw*ai  # GB/s * flop/byte = GFLOP/s
    model_perf=min(memory_roof,MODEL_COMPUTE_CEILING_GFLOPS)
    limit='memory' if memory_roof<MODEL_COMPUTE_CEILING_GFLOPS else 'compute'
    rows.append((name,f'{ai:.3f}',f'{memory_roof:.1f}',f'{model_perf:.1f}',limit))

print(f'Bandwidth input ({"measured useful throughput" if stream else "illustrative fallback"}): {measured_bw:.1f} GB/s')
print(f'Illustrative compute ceiling: {MODEL_COMPUTE_CEILING_GFLOPS:.0f} GFLOP/s')
print()
show_rows(['Kernel','AI flop/B','BW×AI GF/s','Model ceiling GF/s','Likely limit'],rows)


### Explain

At low arithmetic intensity, raising arithmetic throughput alone cannot help much: the memory roof is lower.

As reuse increases, arithmetic intensity moves to the right and the kernel can become less bandwidth-limited.

This is the only Roofline idea you need today:

> **move less data, reuse it sooner, or do more useful work per byte moved.**

A full Roofline analysis requires more careful measurement and is not the goal of this session.


# 9 — Extension: blocking / tiling reuses a small working set

The lab compares two matrix-transpose implementations:

```text
naive:    sweep the full matrix directly
blocked:  work on M3_TILE × M3_TILE tiles before moving on
```

Transpose is used because it makes the locality effect easy to see. The same blocking idea is used in matrix multiplication, image processing and stencil codes.


In [ ]:
rows=kv_lines(M3S2_OUTPUT,'TILING')
if rows:
    table=[]
    for r in rows:
        if 'mode' in r:
            table.append((r['mode'],r.get('block','-'),r['n'],f"{float(r['seconds']):.4f}"))
    show_rows(['Mode','Block','N','Time (s)'],table)
    for r in rows:
        if 'speedup' in r:
            print(f"\nObserved naive/blocked speedup: {float(r['speedup']):.2f}x")
else:
    print('Run Section 2 first.')


### Explain

Blocking does not reduce the number of matrix elements that must be processed.

It changes **when** they are revisited so that useful data has a better chance of remaining in cache.

> **Reorder work to increase reuse before data is evicted.**


### Change, measure, explain

Change only M3_TILE: try 8, 32, 128. Also try M3_MATRIX_N=2049: partial edge tiles must remain correct. Transpose demonstrates locality, not matrix-multiplication performance.

**Your answer:** I changed ___; the measured result changed from ___ to ___; my explanation is ___; an alternative explanation is ___.


In [ ]:
SHOW_C_CODE=True  # Set True to see the actual measured function, including OpenMP directives.
if SHOW_C_CODE: show_kernel('bench_tiling')


# 10 — Extension: loop fusion can remove memory traffic

Two-pass version:

```c
for (i=0; i<N; ++i) temp[i] = A[i] + B[i];
for (i=0; i<N; ++i) out[i]  = temp[i] * scale;
```

Fused version:

```c
for (i=0; i<N; ++i) out[i] = (A[i] + B[i]) * scale;
```

Same mathematical result. The fused version can avoid writing and rereading a very large temporary array.


In [ ]:
rows=kv_lines(M3S2_OUTPUT,'FUSION')
if rows:
    table=[]
    for r in rows:
        if 'mode' in r:
            table.append((r['mode'],r['n'],r['reps'],f"{float(r['seconds']):.4f}"))
    show_rows(['Mode','N','Repetitions','Time (s)'],table)
    for r in rows:
        if 'speedup' in r:
            print(f"\nObserved two-pass/fused speedup: {float(r['speedup']):.2f}x")
else:
    print('Run Section 2 first.')


### Explain

Loop fusion can improve **temporal locality** and reduce memory traffic.

Loop fission can be useful in the opposite situation when one loop's working set is too large or independent phases interfere with cache behavior.

Neither transformation is universally better. The relevant question is:

> **What happens to the working set and bytes moved?**


### Change, measure, explain

Change M3_ARRAY_N and compare the climate preprocessing versions. Two-pass produces a temporary; fused avoids it. Fission is the reverse transformation and is useful only when dependencies permit it.

**Your answer:** I changed ___; the measured result changed from ___ to ___; my explanation is ___; an alternative explanation is ___.


In [ ]:
SHOW_C_CODE=True  # Set True to see the actual measured function, including OpenMP directives.
if SHOW_C_CODE: show_kernel('bench_fusion')


# 11 — Extension: NUMA and first touch

Shared memory does not necessarily mean equal access cost.

On a NUMA node, physical memory is associated with memory domains. Linux commonly places a page near the thread that **first touches** it.

The experiment compares:

1. serial initialization, then many OpenMP threads read the array;
2. parallel initialization with the same static partition used by the later read.

### Predict

On a multi-NUMA-node system, which initialization better matches later parallel ownership?


In [ ]:
print('Compute-node topology:')
print(section(M3S2_OUTPUT,'TOPOLOGY') or 'No topology loaded.')
print()
rows=kv_lines(M3S2_OUTPUT,'FIRST_TOUCH')
if rows:
    table=[]
    for r in rows:
        if 'init' in r:
            table.append((r['init'],r['n'],r['reps'],f"{float(r['seconds']):.4f}"))
    show_rows(['Initialization','N','Read reps','Time (s)'],table)
    for r in rows:
        if 'serial_over_parallel' in r:
            print(f"\nSerial-touch / parallel-touch read time: {float(r['serial_over_parallel']):.2f}x")
else:
    print('Run Section 2 first.')


### Explain

If the node has only one NUMA domain, the two measurements may be similar.

If threads span multiple NUMA domains, parallel first touch can place pages closer to the threads that later consume them.

This is the important principle:

> **where a thread runs and where its data lives can both matter.**

Do not memorize pinning commands here. The point is placement, not command syntax.


### Change, measure, explain

Compare one-worker with 16-worker results. Fresh mmap pages avoid previously placed heap pages. CPU binding spans the allocated cores only; if they are all in one NUMA domain, this cannot demonstrate remote-node placement. No page-residency counters are collected.

**Your answer:** I changed ___; the measured result changed from ___ to ___; my explanation is ___; an alternative explanation is ___.


In [ ]:
SHOW_C_CODE=True  # Set True to see the actual measured function, including OpenMP directives.
if SHOW_C_CODE: show_kernel('bench_first_touch')


# 12 — Extension: vectorization evidence from the compiler

The job also asks GCC for a vectorization report on two loops.

### Loop A — independent iterations

```c
for (i=0; i<n; ++i)
    y[i] = a*x[i] + y[i];
```

### Loop B — loop-carried dependency

```c
for (i=1; i<n; ++i)
    x[i] = x[i-1] + a[i];
```

### Predict

Which loop is easier for SIMD/vectorization?


In [ ]:
report=section(M3S2_OUTPUT,'VECTOR REPORT')
print(report if report else 'No compiler report loaded. Run Section 2 first.')


### Explain

SIMD works best when multiple iterations can execute independently.

The recurrence uses the result of iteration `i-1` in iteration `i`, so the compiler has a real dependency to respect.

> **Vector width does not remove algorithmic dependencies.**

Also note that compiler messages are evidence about what the compiler generated — not proof that a loop is the application's bottleneck. Profiling comes next session.


### Change, measure, explain

Plain and unrolled loops both compile with -O3: the compiler may optimise both similarly. Manual unrolling is not guaranteed to improve speed. Change M3_ARRAY_N to 1048579 to test the leftover elements.

**Your answer:** I changed ___; the measured result changed from ___ to ___; my explanation is ___; an alternative explanation is ___.


In [ ]:
SHOW_C_CODE=True  # Set True to see the actual measured function, including OpenMP directives.
if SHOW_C_CODE: show_kernel('bench_unroll')


# 13 — Extension: sparse structures store what exists

A dense `N×N` matrix stores every value, including zeros.

The CSR-style experiment builds a synthetic matrix with:

- **M3_SPARSE_N rows**;
- **M3_NNZ_PER_ROW stored values per row**;
- arrays for values, column indices and row pointers.

It then performs a real sparse matrix-vector multiply.


In [ ]:
rows=kv_lines(M3S2_OUTPUT,'SPARSE')
if rows:
    r=rows[0]
    show_rows(
        ['Rows','Nonzeros','NNZ/row','CSR MiB','Dense TiB','SpMV time (s)'],
        [(r['n'],r['nnz'],r['nnz_per_row'],r['csr_MiB'],r['dense_TiB'],f"{float(r['seconds']):.4f}")]
    )
else:
    print('Run Section 2 first.')


### Explain

Compressed sparse storage can reduce memory footprint and bytes moved by orders of magnitude when the problem is genuinely sparse.

But CSR introduces:

- index arrays;
- indirect access such as `x[col[j]]`;
- potentially irregular memory behavior.

> **Sparse is not automatically faster. It trades regularity for compression.**


### Change, measure, explain

Change only M3_NNZ_PER_ROW from 5 to 9. The program compares every output row with a serial reference. The dense storage number is calculated, never allocated.

**Your answer:** I changed ___; the measured result changed from ___ to ___; my explanation is ___; an alternative explanation is ___.


In [ ]:
SHOW_C_CODE=True  # Set True to see the actual measured function, including OpenMP directives.
if SHOW_C_CODE: show_kernel('bench_sparse')


# 14 — Extension: software prefetching is a hypothesis, not a guarantee

The final C experiment compares a sequential streaming sum with and without `__builtin_prefetch`.

Modern CPUs already have sophisticated **hardware prefetchers**. Therefore manual prefetching may help, do nothing, or even hurt.

### Predict

Should we assume software prefetching is faster before measuring it?


In [ ]:
rows=kv_lines(M3S2_OUTPUT,'PREFETCH')
if rows:
    table=[]
    for r in rows:
        if 'mode' in r:
            table.append((r['mode'],r.get('distance','-'),r['n'],f"{float(r['seconds']):.4f}"))
    show_rows(['Mode','Distance','N','Time (s)'],table)
    for r in rows:
        if 'speedup_none_over_software' in r:
            print(f"\nNo-prefetch / software-prefetch runtime: {float(r['speedup_none_over_software']):.2f}x")
else:
    print('Run Section 2 first.')


### Explain

If software prefetching does not improve this run, that is a useful result.

A sequential stream is exactly the sort of pattern hardware prefetchers already understand well.

The broader optimization rule is:

> **Predict a mechanism, measure it, and keep the change only if the evidence supports it.**


### Change, measure, explain

Change only M3_PREFETCH_DISTANCE from 64 to 256. Do not assume any improvement: the scalar sum and hardware prefetcher also influence timing.

**Your answer:** I changed ___; the measured result changed from ___ to ___; my explanation is ___; an alternative explanation is ___.


In [ ]:
SHOW_C_CODE=True  # Set True to see the actual measured function, including OpenMP directives.
if SHOW_C_CODE: show_kernel('bench_prefetch')


# 15 — Locality extends beyond one cache: partition data near the computation

The same principle appears at several scales:

```text
cache locality → NUMA locality → node/data locality
```

A 2D domain decomposition gives each MPI rank a local block plus halo cells.

As we create more partitions for the same global grid, each rank owns less interior work while boundary data becomes relatively more important.

This is a MODEL that reconnects to the stencil/domain-decomposition ideas from M3.S1.


In [ ]:
import math

global_n=4096
parts=[1,4,16,64]
rows=[]
for p in parts:
    side=int(round(math.sqrt(p)))
    local=global_n//side
    interior=local*local
    halo=0 if p==1 else 4*local
    ratio=halo/interior
    rows.append((p,f'{side}×{side}',f'{local}×{local}',interior,halo,f'{100*ratio:.3f}%'))

show_rows(['MPI ranks','Partition grid','Local block','Interior cells','Halo cells*','Halo/Interior'],rows)
print('\n* Simple 1-cell halo model; corners and implementation details omitted.')


### Explain

Good partitioning aims to keep most computation on locally owned data and communicate only what is necessary.

The same design question keeps recurring:

> **Can we keep the data close to the computation that needs it?**


# 16 — Five memory-performance traps

For each case, name the most likely issue **before** opening the answer.

### A — Large array, stride = 8 doubles
One useful value is consumed from many cache lines.

<details><summary><strong>Reveal</strong></summary>
Poor spatial locality / wasted cache-line transfer.
</details>

### B — 16 threads update adjacent private counters
No data race, but scaling collapses.

<details><summary><strong>Reveal</strong></summary>
False sharing / coherence traffic.
</details>

### C — 100 million particles, hot loop touches only `x`
Each particle object is 64 bytes.

<details><summary><strong>Reveal</strong></summary>
Data-layout mismatch; investigate SoA or another layout that keeps the hot field contiguous.
</details>

### D — Memory bandwidth is already saturated
More threads barely improve runtime.

<details><summary><strong>Reveal</strong></summary>
Bandwidth bottleneck; more arithmetic resources alone do not solve it.
</details>

### E — Two-socket NUMA node, serial initialization
Later threads run on both sockets.

<details><summary><strong>Reveal</strong></summary>
Possible remote memory access due to first-touch placement.
</details>


# 17 — Exit ticket

Answer each in one or two sentences.

### 1. What is the difference between latency and bandwidth?

Your answer:

### 2. Why can row-major and column-major traversal differ even with identical arithmetic?

Your answer:

### 3. What is false sharing?

Your answer:

### 4. When would SoA be worth investigating?

Your answer:

### 5. What does arithmetic intensity tell us in Roofline reasoning?

Your answer:

### 6. Why can first-touch initialization affect performance on NUMA systems?

Your answer:

<details>
<summary><strong>Show concise answers</strong></summary>

1. **Latency** is delay per request; **bandwidth** is sustained transfer rate.
2. Traversal order changes spatial locality and cache-line reuse.
3. Independent variables on the same cache line can trigger coherence invalidations when different cores write them.
4. When a hot kernel uses only a subset of fields and contiguous field arrays could move less irrelevant data or vectorize better.
5. It is useful arithmetic per byte moved; low intensity tends to expose memory bandwidth more strongly.
6. Pages are commonly placed near the thread that first touches them, so serial initialization can concentrate memory on one NUMA domain.

</details>


# What you should leave with

- Memory is **hierarchical**: registers → caches → DRAM.
- **Latency** and **bandwidth** describe different limits.
- Cache lines make **spatial locality** valuable.
- Reuse makes **temporal locality** valuable.
- Same Big-O complexity can have very different memory behavior.
- More threads can hit a **memory-bandwidth ceiling**.
- **False sharing** is a layout/coherence problem, not a data-race problem.
- **AoS vs SoA** should be chosen from the hot access pattern, not by slogan.
- Blocking and fusion can reduce repeated movement through memory.
- **NUMA placement** means shared memory may not be equally near.
- Sparse formats can save enormous storage and movement, but introduce indirect access.
- Qualitative Roofline reasoning asks how much useful work is done per byte moved.

The unifying goal is:

> **move less data, move it predictably, reuse it sooner.**

Next: **M3.S3 — Profiling and Performance Analysis**: turn these hypotheses into measured bottleneck evidence.


# 18 — Loop unrolling: does the compiler already do it?

Compare a normal vector-add loop with four explicit additions per iteration plus a safe tail loop. Same values, same optimisation flags. This measures the result of source and compiler together.


In [ ]:
rows=kv_lines(M3S2_OUTPUT,"UNROLL")
show_rows(["Mode","N","Seconds"],[(r["mode"],r["n"],r["seconds"]) for r in rows if "mode" in r])
SHOW_UNROLL_CODE=True
if SHOW_UNROLL_CODE: show_kernel("bench_unroll")


# 19 — Small correctness lab: climate data, image tiles and sparse neighbours

These Python activities run anywhere. They illustrate layout and transformations, **not hardware performance**. Change the marked inputs and rerun.


In [ ]:
temperatures=[12.0,15.0,18.0]  # EDIT: add a location
corrections=[0.5,-1.0,2.0]    # Keep one correction per location
scale=1.8
if len(temperatures)!=len(corrections): raise ValueError("One correction per location")
temp=[a+b for a,b in zip(temperatures,corrections)]
two_pass=[v*scale for v in temp]
fused=[(a+b)*scale for a,b in zip(temperatures,corrections)]
assert two_pass==fused
print("Climate preprocessing:",fused,"— identical answers, fused has no temporary list")
image=[[1,2,3],[4,5,6]]  # EDIT dimensions and values
transpose=[[image[i][j] for i in range(len(image))] for j in range(len(image[0]))]
print("Image rows:",image,"transposed:",transpose)
# CSR matrix [[2,0,1],[0,3,0],[4,0,5]]
rowptr=[0,2,3,5];cols=[0,2,1,0,2];values=[2,1,3,4,5]
x=[10,20,30]  # EDIT input values
assert rowptr[-1]==len(cols)==len(values)
y=[sum(values[j]*x[cols[j]] for j in range(rowptr[i],rowptr[i+1])) for i in range(3)]
dense=[[2,0,1],[0,3,0],[4,0,5]]
reference=[sum(a*b for a,b in zip(row,x)) for row in dense]
assert y==reference
print("Sparse interaction result:",y,"— matches dense reference")


# 20 — Compare two real experiments

Run defaults, change ONE input, then enter both job IDs. Use measurements to support a claim; never invent data when the queue is unavailable. Short and fixed-order measurements can be noisy: repeat unchanged settings before claiming small improvements. Cache-miss counts, energy consumption, GPU memory and network performance are not measured.


In [ ]:
BEFORE_JOB_ID=None
AFTER_JOB_ID=None
if BEFORE_JOB_ID and AFTER_JOB_ID:
    records=[json.loads((DEMO_DIR/f"experiment-{j}.json").read_text()) for j in [BEFORE_JOB_ID,AFTER_JOB_ID]]
    changes={k:(records[0]["settings"][k],records[1]["settings"][k]) for k in LAB_SETTINGS if records[0]["settings"][k]!=records[1]["settings"][k]}
    print("Changed settings:",changes)
    for key in ["STREAM","MATRIX","TILING","AOSSOA"]:
        print("\n",key)
        for record in records:
            print("Job",record["job_id"],kv_lines(record["output"],key))
else: print("Enter two completed job IDs after changing one input.")
